# Experiment 1: the plan's tables, recomputed from the run files

Every table below is built **in this notebook** from `out/*.jsonl`. No
table-building function is imported. The statistics, the subsetting and the
denominators are all written out in Setup so they can be read and changed.

The last section cross-checks the results against `ex1_results.py`. If the two
disagree, one of them is wrong and neither should go in the chapter.

| Table | What it answers | Plan section |
|---|---|---|
| T0 | Reference lines | Reference lines |
| T1 | Width x name removal, three measures | Design 1 |
| T2 | Contrasts and the interaction | Design 1 |
| T3 | Width x name truth, legality and refusal | Design 2 |
| T4 | The directional swap test | Design 2 |
| T5 | Rule removal, with the empty cell marked | Design 3 |
| T6 | Violation composition | Design 3 |
| T7 | Three signatures of an unregistered loss | Calibration |
| T8 | Cast B | not in the plan, recommended |

## Setup: paths, loading, and every statistic used below

In [12]:
import os, sys, json, math, collections, pathlib

# --- find the package root (the directory holding out/ and probes/) ----------
here = pathlib.Path.cwd()
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "probes").is_dir():
        ROOT = str(cand)
        break
else:
    raise SystemExit("package root not found: no directory above holds out/ and probes/")

print("root", ROOT)

MODELS = ["gemini", "gpt", "qwen"]
LABEL  = {"gemini": "Gemini", "gpt": "GPT", "qwen": "Qwen"}

# Condition -> the word in the file name. Repeats are 3 everywhere except
# Legal-Arm Control, which was run once.
FILENAME = {
    "full":          "ex1_casta_{m}_full_r3.jsonl",
    "anon":          "ex1_casta_{m}_anon_r3.jsonl",
    "swap":          "ex1_casta_{m}_swap_r3.jsonl",
    "nowidth":       "ex1_casta_{m}_nowidth_r3.jsonl",
    "nowidth-anon":  "ex1_casta_{m}_nowidth-anon_r3.jsonl",
    "nowidth-swap":  "ex1_casta_{m}_nowidth-swap_r3.jsonl",
    "norules":       "ex1_casta_{m}_norules_r3.jsonl",
    "givenset":      "ex1_casta_{m}_givenset_r1.jsonl",
}
PRETTY = {"full": "Full information", "anon": "Name removed",
          "swap": "Name swapped", "nowidth": "Width removed",
          "nowidth-anon": "Both removed", "nowidth-swap": "Width removed + swapped",
          "norules": "No rules", "givenset": "Legal set printed"}

_cache = {}
def rows(model, cond):
    # Every row of one run file. Cached, because these are read many times.
    key = (model, cond)
    if key not in _cache:
        path = os.path.join(ROOT, "out", FILENAME[cond].format(m=model))
        with open(path) as f:
            _cache[key] = [json.loads(l) for l in f if l.strip()]
    return _cache[key]

root /Users/erinsarlak/Downloads/MastersDissertation/fourarm


In [13]:
# --- statistics -------------------------------------------------------------

def wilson(k, n, z=1.96):
    # Wilson score interval for a proportion, as percentages.
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, centre - half), 100 * min(1.0, centre + half))


def newcombe(k1, n1, k2, n2, z=1.96):
    # Group 1 minus group 2, Newcombe hybrid-score interval, as percentages.
    if n1 == 0 or n2 == 0:
        return (float("nan"),) * 3
    l1, u1 = (x / 100 for x in wilson(k1, n1, z))
    l2, u2 = (x / 100 for x in wilson(k2, n2, z))
    p1, p2 = k1 / n1, k2 / n2
    d = p1 - p2
    return (100 * d,
            100 * (d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)),
            100 * (d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)))


def diff_of_diffs(a, b):
    # One Newcombe difference minus another. Normal approximation.
    se = math.sqrt(((a[2] - a[1]) / 3.92) ** 2 + ((b[2] - b[1]) / 3.92) ** 2)
    d = a[0] - b[0]
    return d, d - 1.96 * se, d + 1.96 * se


def pct(k, n):
    return float("nan") if n == 0 else 100.0 * k / n

def ci(k, n):
    lo, hi = wilson(k, n)
    return "%.1f [%.1f, %.1f]" % (pct(k, n), lo, hi)

def gap(base, manip):
    # Cost of the manipulation: base minus manipulated, with an interval.
    d, lo, hi = newcombe(base[0], base[1], manip[0], manip[1])
    return "%+.1f [%+.1f, %+.1f]" % (d, lo, hi)

def spans_zero(t):
    return t[1] <= 0 <= t[2]

In [14]:
# --- measures ---------------------------------------------------------------
# Each returns (numerator, denominator) so intervals can be taken on it.

def scene_key(r):
    p = r["provenance"]
    return (p["source"], p["seq"], p["round"])


def legality(rs, grasp=True):
    # Legality on the grasp-binding states. grasp=False is the negative
    # control. Restricted to states with at least one legal option, because a
    # refusal state has no correct assignment to score.
    k = n = 0
    for r in rs:
        if r.get("zero_legal") or bool(r.get("binds_grasp")) != grasp:
            continue
        if r.get("result") not in ("valid", "rejected"):
            continue
        n += 1
        k += r["result"] == "valid"
    return k, n


def legality_scene(rs, grasp=True):
    # Per-scene majority. 288 trials are 96 scenes by 3 repeats.
    g = collections.defaultdict(list)
    for r in rs:
        if r.get("zero_legal") or bool(r.get("binds_grasp")) != grasp:
            continue
        if r.get("result") not in ("valid", "rejected"):
            continue
        g[scene_key(r)].append(r["result"] == "valid")
    return sum(1 for v in g.values() if sum(v) * 2 > len(v)), len(g)


def refusal(rs):
    # Correct refusal, trial level, over the states with no legal option.
    k = n = 0
    for r in rs:
        if not r.get("zero_legal"):
            continue
        n += 1
        k += r.get("result") == "noop"
    return k, n


def refusal_scene(rs):
    # Correct refusal at the scene denominator. This one is authoritative.
    g = collections.defaultdict(list)
    for r in rs:
        if not r.get("zero_legal"):
            continue
        g[scene_key(r)].append(r.get("result") == "noop")
    return sum(1 for v in g.values() if sum(v) * 2 > len(v)), len(g)


def franka(rs):
    # Share of proposals sent to a Franka. Two Franka and two UR, so 50 is
    # indifference between the arm types.
    k = n = 0
    for r in rs:
        arm = (r.get("decision") or {}).get("arm")
        if not arm:
            continue
        n += 1
        k += arm.startswith("franka")
    return k, n


def violations(rs):
    return collections.Counter(r.get("violation_cause") for r in rs
                               if r.get("violation_cause"))

print("measures defined")

measures defined


In [15]:
# --- display ----------------------------------------------------------------
try:
    import pandas as pd
    pd.set_option("display.max_colwidth", 60)
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False

try:
    from IPython.display import display
except ImportError:
    display = None

def table(headers, body, title=""):
    # Renders in place, so text may follow the table in the same cell.
    if HAVE_PANDAS and display is not None:
        if title:
            print(title)
        display(pd.DataFrame(body, columns=headers))
        return
    w = [max(len(str(h)), *(len(str(r[i])) for r in body))
         for i, h in enumerate(headers)]
    line = "  ".join("-" * x for x in w)
    if title:
        print("\n" + title)
    print(line)
    print("  ".join(str(h).ljust(x) for h, x in zip(headers, w)))
    print(line)
    for r in body:
        print("  ".join(str(c).ljust(x) for c, x in zip(r, w)))
    print(line)

# Sanity: the files load and are the size they should be.
for m in MODELS:
    for c in FILENAME:
        n = len(rows(m, c))
        want = 162 if c == "givenset" else 486
        flag = "" if n == want else "   <-- expected %d" % want
        if flag:
            print("%-7s %-14s %d rows%s" % (LABEL[m], c, n, flag))
print("all files loaded")

all files loaded


---

# T0. Reference lines

A raw legality score means nothing without anchors. Two are computed offline
over the probe set and stored in `out/ex1_chance_floor.json`.

- **Chance floor.** Guessing uniformly among the arms.
- **Width-blind line.** Applying every constraint except grasp perfectly, then
  guessing among the arms that survive. This is the score of a model that has
  lost exactly the width and nothing else.

Both are reported on two populations: all picking states, and the grasp-binding
subset. That the width-blind line is *lower* on the grasp-binding subset is what
shows the subset is where the width bites.

In [16]:
floors = json.load(open(os.path.join(ROOT, "out", "ex1_chance_floor.json")))

body = [
    ["All picking states", floors["n_picking"],
     "%.1f" % (100 * floors["uniform"]["mean"]),
     "%.1f" % (100 * floors["width_blind"]["mean"])],
    ["Grasp-binding subset", floors["by_cause"]["grasp"]["n_states"],
     "%.1f" % (100 * floors["by_cause"]["grasp"]["uniform"]["mean"]),
     "%.1f" % (100 * floors["by_cause"]["grasp"]["width_blind"]["mean"])],
]
CHANCE = 100 * floors["by_cause"]["grasp"]["uniform"]["mean"]
WIDTH_BLIND = 100 * floors["by_cause"]["grasp"]["width_blind"]["mean"]
N_REFUSAL = floors["n_refusal"]

table(["Population", "States", "Chance floor", "Width-blind line"], body,
      "T0  Reference lines, cast A")
print("\nprobe set %s, %d states, %d picking, %d with no legal arm"
      % (floors["probe_set"], floors["n_states"], floors["n_picking"], N_REFUSAL))
print("Legality below is measured on the %d grasp-binding states against %.1f."
      % (floors["by_cause"]["grasp"]["n_states"], WIDTH_BLIND))
print("Refusal is measured on the %d no-legal-arm states. Different question,"
      % N_REFUSAL)
print("different population, different anchor.")


T0  Reference lines, cast A
--------------------  ------  ------------  ----------------
Population            States  Chance floor  Width-blind line
--------------------  ------  ------------  ----------------
All picking states    126     35.5          80.9            
Grasp-binding subset  96      30.5          74.9            
--------------------  ------  ------------  ----------------

probe set ex1_v2.json, 162 states, 126 picking, 36 with no legal arm
Legality below is measured on the 96 grasp-binding states against 74.9.
Refusal is measured on the 36 no-legal-arm states. Different question,
different population, different anchor.


---

# T1. Design 1: remove the width, remove the name

The 2x2. Read top to bottom for the width effect, left to right for the name
effect.

|  | Name kept | Name removed |
|---|---|---|
| Width kept | `full` | `anon` |
| Width removed | `nowidth` | `nowidth-anon` |

Three measures across the same four cells, so one manipulation is seen moving
several behaviours together. Legality says whether the model got it right.
Refusal and Franka share say *how* it fails.

In [17]:
GRID1 = ["full", "anon", "nowidth", "nowidth-anon"]

body = []
for m in MODELS:
    for name, fn, unit in (("Legality (grasp-binding)", legality, "trial"),
                           ("Legality (scene majority)", legality_scene, "scene"),
                           ("Correct refusal", refusal, "trial"),
                           ("Franka share", franka, "proposal")):
        r = [LABEL[m], name]
        for c in GRID1:
            k, n = fn(rows(m, c))
            r.append("%.1f  (%d/%d)" % (pct(k, n), k, n))
        body.append(r)

table(["Model", "Measure"] + [PRETTY[c] for c in GRID1], body,
      "T1  Design 1, all four cells, cast A")
print("\nwidth-blind line %.1f, chance floor %.1f" % (WIDTH_BLIND, CHANCE))


T1  Design 1, all four cells, cast A
------  -------------------------  ----------------  ----------------  ---------------  ---------------
Model   Measure                    Full information  Name removed      Width removed    Both removed   
------  -------------------------  ----------------  ----------------  ---------------  ---------------
Gemini  Legality (grasp-binding)   100.0  (288/288)  100.0  (288/288)  72.2  (208/288)  75.3  (217/288)
Gemini  Legality (scene majority)  100.0  (96/96)    100.0  (96/96)    71.9  (69/96)    77.1  (74/96)  
Gemini  Correct refusal            93.5  (101/108)   93.5  (101/108)   50.9  (55/108)   51.9  (56/108) 
Gemini  Franka share               28.1  (108/385)   27.8  (107/385)   48.0  (207/431)  46.0  (198/430)
GPT     Legality (grasp-binding)   97.5  (276/283)   98.6  (276/280)   73.6  (206/280)  76.7  (217/283)
GPT     Legality (scene majority)  97.9  (94/96)     98.9  (94/95)     71.6  (68/95)    77.9  (74/95)  
GPT     Correct refusal   

In [18]:
# Does each width-removed cell sit ON the width-blind line, or merely near it?
# The reference line is the strongest device in the chapter, so test it.
print("Grasp-binding legality against the width-blind line of %.1f\n" % WIDTH_BLIND)
for m in MODELS:
    for c in ("nowidth", "nowidth-anon"):
        k, n = legality(rows(m, c))
        lo, hi = wilson(k, n)
        print("  %-7s %-14s %5.1f [%5.1f, %5.1f]  %s"
              % (LABEL[m], c, pct(k, n), lo, hi,
                 "includes the line" if lo <= WIDTH_BLIND <= hi
                 else "EXCLUDES the line"))

Grasp-binding legality against the width-blind line of 74.9

  Gemini  nowidth         72.2 [ 66.8,  77.1]  includes the line
  Gemini  nowidth-anon    75.3 [ 70.1,  80.0]  includes the line
  GPT     nowidth         73.6 [ 68.1,  78.4]  includes the line
  GPT     nowidth-anon    76.7 [ 71.4,  81.2]  includes the line
  Qwen    nowidth         76.7 [ 71.5,  81.2]  includes the line
  Qwen    nowidth-anon    72.6 [ 67.1,  77.4]  includes the line


---

# T2. Design 1 contrasts, and the interaction

The grid shows levels. This shows the differences, which is what the prose
claims. Base minus manipulated, so a **positive value is the cost** of the
manipulation.

The interaction is the width effect with the name present minus the width effect
with the name removed. If the name were carrying width information, removing it
would make the width effect smaller, and the interaction would be positive.

In [19]:
CONTRASTS1 = [
    ("Width removed", "name kept",    "full",         "nowidth"),
    ("Width removed", "name removed", "anon",         "nowidth-anon"),
    ("Name removed",  "width kept",   "full",         "anon"),
    ("Name removed",  "width removed","nowidth",      "nowidth-anon"),
]

body = []
for fam, lvl, base, manip in CONTRASTS1:
    r = [fam, lvl]
    for m in MODELS:
        r.append(gap(legality(rows(m, base)), legality(rows(m, manip))))
    body.append(r)

table(["Family", "Level"] + [LABEL[m] for m in MODELS], body,
      "T2  Design 1 contrasts, grasp-binding legality, trial level")


T2  Design 1 contrasts, grasp-binding legality, trial level
-------------  -------------  --------------------  --------------------  ------------------
Family         Level          Gemini                GPT                   Qwen              
-------------  -------------  --------------------  --------------------  ------------------
Width removed  name kept      +27.8 [+22.7, +33.2]  +24.0 [+18.5, +29.6]  -1.4 [-8.3, +5.6] 
Width removed  name removed   +24.7 [+19.8, +29.9]  +21.9 [+16.8, +27.2]  +4.2 [-2.9, +11.2]
Name removed   width kept     +0.0 [-1.3, +1.3]     -1.0 [-3.7, +1.5]     -1.4 [-8.3, +5.6] 
Name removed   width removed  -3.1 [-10.3, +4.1]    -3.1 [-10.2, +4.0]    +4.2 [-2.9, +11.2]
-------------  -------------  --------------------  --------------------  ------------------


In [20]:
# The same contrasts at the scene denominator. A conclusion that changes between
# the two is not a conclusion.
print("scene denominator, and whether any conclusion changes\n")
changed = 0
for fam, lvl, base, manip in CONTRASTS1:
    line = "  %-14s %-14s" % (fam, lvl)
    for m in MODELS:
        t = newcombe(*legality(rows(m, base)), *legality(rows(m, manip)))
        s = newcombe(*legality_scene(rows(m, base)), *legality_scene(rows(m, manip)))
        flip = spans_zero(t) != spans_zero(s)
        changed += flip
        line += "  %-7s %+5.1f [%+5.1f, %+5.1f]%s" % (
            LABEL[m], s[0], s[1], s[2], " CHANGED" if flip else "")
    print(line)
print("\ncells whose conclusion changes between denominators: %d of %d"
      % (changed, len(CONTRASTS1) * len(MODELS)))

scene denominator, and whether any conclusion changes

  Width removed  name kept       Gemini  +28.1 [+19.2, +37.8]  GPT     +26.3 [+16.7, +36.2]  Qwen     -1.0 [-12.8, +10.7]
  Width removed  name removed    Gemini  +22.9 [+14.7, +32.3]  GPT     +21.1 [+12.5, +30.4]  Qwen     +3.1 [ -9.2, +15.3]
  Name removed   width kept      Gemini   +0.0 [ -3.8,  +3.8]  GPT      -1.0 [ -6.3,  +3.9]  Qwen     +1.0 [-10.9, +13.0]
  Name removed   width removed   Gemini   -5.2 [-17.3,  +7.1]  GPT      -6.3 [-18.4,  +6.0]  Qwen     +5.2 [ -7.0, +17.2]

cells whose conclusion changes between denominators: 0 of 12


In [21]:
# The interaction, with its resolution. A null without a stated resolution is
# not a result.
print("Interaction: width effect with the name, minus width effect without it\n")
for m in MODELS:
    a = newcombe(*legality(rows(m, "full")),  *legality(rows(m, "nowidth")))
    b = newcombe(*legality(rows(m, "anon")),  *legality(rows(m, "nowidth-anon")))
    d = diff_of_diffs(a, b)
    print("  %-7s %+5.1f [%+5.1f, %+5.1f]   resolution about %.0f points   %s"
          % (LABEL[m], d[0], d[1], d[2], (d[2] - d[1]) / 2,
             "spans zero" if spans_zero(d) else "EXCLUDES ZERO"))

Interaction: width effect with the name, minus width effect without it

  Gemini   +3.1 [ -4.1, +10.4]   resolution about 7 points   spans zero
  GPT      +2.1 [ -5.5,  +9.6]   resolution about 8 points   spans zero
  Qwen     -5.6 [-15.5,  +4.4]   resolution about 10 points   spans zero


---

# T3. Design 2: keep the width, lie about the name

|  | Name true | Name false |
|---|---|---|
| Width kept | `full` | `swap` |
| Width removed | `nowidth` | `nowidth-swap` |

The parent of `nowidth-swap` is `nowidth`, not `nowidth-anon`, because the
swapped condition carries names that are present but false.

The plan promises this reported on legality **and refusal**, so both are here.

In [22]:
GRID2 = ["full", "swap", "nowidth", "nowidth-swap"]

body = []
for m in MODELS:
    for name, fn in (("Legality (grasp-binding)", legality),
                     ("Correct refusal", refusal)):
        r = [LABEL[m], name]
        for c in GRID2:
            k, n = fn(rows(m, c))
            r.append("%.1f  (%d/%d)" % (pct(k, n), k, n))
        body.append(r)
table(["Model", "Measure"] + [PRETTY[c] for c in GRID2], body,
      "T3  Design 2, all four cells, cast A")


T3  Design 2, all four cells, cast A
------  ------------------------  ----------------  ----------------  ---------------  -----------------------
Model   Measure                   Full information  Name swapped      Width removed    Width removed + swapped
------  ------------------------  ----------------  ----------------  ---------------  -----------------------
Gemini  Legality (grasp-binding)  100.0  (288/288)  100.0  (288/288)  72.2  (208/288)  76.0  (218/287)        
Gemini  Correct refusal           93.5  (101/108)   95.4  (103/108)   50.9  (55/108)   50.0  (54/108)         
GPT     Legality (grasp-binding)  97.5  (276/283)   97.5  (268/275)   73.6  (206/280)  77.6  (218/281)        
GPT     Correct refusal           88.9  (96/108)    91.7  (99/108)    53.7  (58/108)   55.6  (60/108)         
Qwen    Legality (grasp-binding)  75.3  (217/288)   75.3  (217/288)   76.7  (221/288)  75.0  (216/288)        
Qwen    Correct refusal           0.0  (0/108)      0.0  (0/108)      0.0 

In [23]:
body = []
for fam, base, manip in (("Name swapped, width kept",    "full",    "swap"),
                         ("Name swapped, width removed", "nowidth", "nowidth-swap")):
    for name, fn in (("legality", legality), ("refusal", refusal)):
        r = [fam, name]
        for m in MODELS:
            r.append(gap(fn(rows(m, base)), fn(rows(m, manip))))
        body.append(r)
table(["Contrast", "Measure"] + [LABEL[m] for m in MODELS], body,
      "T3b  Design 2 contrasts, base minus swapped")


T3b  Design 2 contrasts, base minus swapped
---------------------------  --------  -------------------  -------------------  -----------------
Contrast                     Measure   Gemini               GPT                  Qwen             
---------------------------  --------  -------------------  -------------------  -----------------
Name swapped, width kept     legality  +0.0 [-1.3, +1.3]    +0.1 [-2.8, +3.0]    +0.0 [-7.0, +7.0]
Name swapped, width kept     refusal   -1.9 [-8.7, +4.8]    -2.8 [-11.1, +5.4]   +0.0 [-3.4, +3.4]
Name swapped, width removed  legality  -3.7 [-10.8, +3.4]   -4.0 [-11.1, +3.1]   +1.7 [-5.2, +8.7]
Name swapped, width removed  refusal   +0.9 [-12.2, +14.0]  -1.9 [-14.9, +11.2]  +0.0 [-3.4, +3.4]
---------------------------  --------  -------------------  -------------------  -----------------


---

# T4. The directional swap test

The test is **not** whether an interval excludes zero. Name-following requires
the two swap directions to move in **opposite** directions:

- a wide object relabelled with a narrow name should go to a Franka *more*,
- a narrow object relabelled with a wide name should go to a Franka *less*.

So the statistic is the separation between them, judged against the drift on the
four objects that were never touched.

In [24]:
APERTURE = 0.080
SWAP_PAIRS = [("ycb_large_clamp", 0.122), ("ycb_mustard", 0.096),
              ("ycb_meat_can", 0.084),   ("ycb_power_drill", 0.050),
              ("ycb_soup_can", 0.068),   ("ycb_gelatin_box", 0.073)]
SWAP_CONTROLS = ["ycb_mug", "ycb_mug2", "ycb_banana", "ycb_bowl"]

# Which object each task really is, taken from the probe set, so a swapped name
# in the prompt cannot confuse the accounting.
probes = json.load(open(os.path.join(ROOT, "probes", "ex1_v2.json")))["probes"]
TRUE_OBJECT = {}
for p in probes:
    pv = p["provenance"]
    for t in p["state"]["tasks"]:
        TRUE_OBJECT[(pv["source"], pv["seq"], pv["round"], t["id"])] = t["object"]

def franka_by_object(rs):
    tot, fr = collections.Counter(), collections.Counter()
    for r in rs:
        dec = r.get("decision") or {}
        arm, tid = dec.get("arm"), dec.get("task_id")
        if not arm or tid is None:
            continue
        pv = r["provenance"]
        obj = TRUE_OBJECT.get((pv["source"], pv["seq"], pv["round"], tid))
        if obj is None:
            continue
        tot[obj] += 1
        fr[obj] += arm.startswith("franka")
    return fr, tot

body = []
for m in MODELS:
    for base, manip, lvl in (("full", "swap", "width kept"),
                             ("nowidth", "nowidth-swap", "width removed")):
        fb, tb = franka_by_object(rows(m, base))
        fs, ts = franka_by_object(rows(m, manip))
        acc = {"wide": [0, 0, 0, 0], "narrow": [0, 0, 0, 0], "control": [0, 0, 0, 0]}
        for obj, w in SWAP_PAIRS:
            g = acc["wide" if w > APERTURE else "narrow"]
            g[0] += fs[obj]; g[1] += ts[obj]; g[2] += fb[obj]; g[3] += tb[obj]
        for obj in SWAP_CONTROLS:
            g = acc["control"]
            g[0] += fs[obj]; g[1] += ts[obj]; g[2] += fb[obj]; g[3] += tb[obj]
        ch = {k: newcombe(*v) for k, v in acc.items()}
        sep = diff_of_diffs(ch["wide"], ch["narrow"])
        body.append([LABEL[m], lvl,
                     "%+.1f" % ch["wide"][0], "%+.1f" % ch["narrow"][0],
                     "%+.1f" % ch["control"][0],
                     "%+.1f [%+.1f, %+.1f]" % sep,
                     "spans zero" if spans_zero(sep) else "EXCLUDES ZERO"])

table(["Model", "Width", "Wide obj, narrow name", "Narrow obj, wide name",
       "Untouched controls", "Separation", ""], body,
      "T4  Change in Franka share by swap direction")
print("\nName-following predicts the first two columns move in OPPOSITE")
print("directions. Compare the separation against the control drift.")


T4  Change in Franka share by swap direction
------  -------------  ---------------------  ---------------------  ------------------  -------------------  ----------
Model   Width          Wide obj, narrow name  Narrow obj, wide name  Untouched controls  Separation                     
------  -------------  ---------------------  ---------------------  ------------------  -------------------  ----------
Gemini  width kept     +0.0                   -5.3                   +0.6                +5.3 [-8.3, +19.0]   spans zero
Gemini  width removed  -5.9                   -6.8                   +0.6                +0.8 [-15.9, +17.5]  spans zero
GPT     width kept     +1.1                   -6.7                   -2.2                +7.8 [-7.4, +23.0]   spans zero
GPT     width removed  -3.4                   -1.6                   +0.3                -1.8 [-17.7, +14.0]  spans zero
Qwen    width kept     +2.3                   -3.0                   +4.7                +5.3 [-4.7, +15.4]

In [25]:
# The single strongest sentence available for this design: legality while every
# swapped name was lying.
for m in MODELS:
    k, n = legality(rows(m, "swap"))
    print("  %-7s grasp-binding legality with names swapped: %d/%d = %.1f"
          % (LABEL[m], k, n, pct(k, n)))
print("\nRun analysis/ex1/ex1_check_swap.py for the evidence that the models")
print("named the swapped partner in their own reasons, which is what shows the")
print("manipulation reached them rather than being ignored at the parser.")

  Gemini  grasp-binding legality with names swapped: 288/288 = 100.0
  GPT     grasp-binding legality with names swapped: 268/275 = 97.5
  Qwen    grasp-binding legality with names swapped: 217/288 = 75.3

Run analysis/ex1/ex1_check_swap.py for the evidence that the models
named the swapped partner in their own reasons, which is what shows the
manipulation reached them rather than being ignored at the parser.


---

# T5. Design 3: remove the rule, not the data

|  | Rule stated | Rule removed |
|---|---|---|
| Width kept | `full` | `norules` |
| Width removed | `nowidth` | *not run* |

**The empty cell.** The fourth cell is not run because a drop from `nowidth` to
`nowidth + norules` would confound instruction loss with the width loss already
measured above it. That is the reason recorded in `experiments/ex1/prompts.py`.

The second cell below tests the *other* justification, that the width-removed
row has no room to fall further. It does not hold, so do not use it.

Note that No Rules withholds **two** rules together, R3 (grasp and delicacy) and
R4 (reach). It is a single manipulation on the rules axis and cannot be
decomposed.

In [26]:
body = []
for m in MODELS:
    kf, nf = legality(rows(m, "full"))
    kr, nr = legality(rows(m, "norules"))
    kw, nw = legality(rows(m, "nowidth"))
    body.append([LABEL[m], ci(kf, nf), ci(kr, nr),
                 gap((kf, nf), (kr, nr)), ci(kw, nw), "not run"])
table(["Model", "Full information", "No rules", "Cost of removing the rules",
       "Width removed", "Width removed + no rules"], body,
      "T5  Design 3, grasp-binding legality")


T5  Design 3, grasp-binding legality
------  -------------------  -------------------  --------------------------  -----------------  ------------------------
Model   Full information     No rules             Cost of removing the rules  Width removed      Width removed + no rules
------  -------------------  -------------------  --------------------------  -----------------  ------------------------
Gemini  100.0 [98.7, 100.0]  100.0 [98.7, 100.0]  +0.0 [-1.3, +1.3]           72.2 [66.8, 77.1]  not run                 
GPT     97.5 [95.0, 98.8]    95.1 [91.9, 97.1]    +2.4 [-0.9, +5.9]           73.6 [68.1, 78.4]  not run                 
Qwen    75.3 [70.1, 80.0]    72.2 [66.8, 77.1]    +3.1 [-4.1, +10.3]          76.7 [71.5, 81.2]  not run                 
------  -------------------  -------------------  --------------------------  -----------------  ------------------------


In [27]:
# Is "no room to fall further" true? Compare the width-removed row against the
# chance floor, not against the width-blind line.
print("width-blind line %.1f, chance floor %.1f, %.1f points between them\n"
      % (WIDTH_BLIND, CHANCE, WIDTH_BLIND - CHANCE))
for m in MODELS:
    k, n = legality(rows(m, "nowidth"))
    print("  %-7s width removed sits at %.1f, %.1f points ABOVE the chance floor"
          % (LABEL[m], pct(k, n), pct(k, n) - CHANCE))
print("\nThere is room to fall. Use the confounding argument, not this one.")

width-blind line 74.9, chance floor 30.5, 44.4 points between them

  Gemini  width removed sits at 72.2, 41.7 points ABOVE the chance floor
  GPT     width removed sits at 73.6, 43.1 points ABOVE the chance floor
  Qwen    width removed sits at 76.7, 46.2 points ABOVE the chance floor

There is room to fall. Use the confounding argument, not this one.


---

# T6. Violation composition

What the rule removal actually changes. A violation on a state with no legal arm
is a failure to refuse. A violation on a picking state is a wrong choice. Pooling
the two hides the difference, so both splits are shown.

In [28]:
CAUSES = ["grasp", "reach", "delicate", "arm_state", "no_route"]
body = []
for m in MODELS:
    for c in ("full", "nowidth", "norules"):
        v = violations(rows(m, c))
        body.append([LABEL[m], PRETTY[c]] + [v.get(x, 0) for x in CAUSES]
                    + [sum(v.values())])
table(["Model", "Condition"] + CAUSES + ["Total"], body,
      "T6  Violations by cause, pooled over all states")


T6  Violations by cause, pooled over all states
------  ----------------  -----  -----  --------  ---------  --------  -----
Model   Condition         grasp  reach  delicate  arm_state  no_route  Total
------  ----------------  -----  -----  --------  ---------  --------  -----
Gemini  Full information  0      0      0         0          7         7    
Gemini  Width removed     133    0      0         0          0         133  
Gemini  No rules          1      0      0         0          0         1    
GPT     Full information  7      0      2         0          10        19   
GPT     Width removed     121    0      0         0          3         124  
GPT     No rules          28     0      3         0          5         36   
Qwen    Full information  41     52     39        67         1         200  
Qwen    Width removed     43     56     34        55         1         189  
Qwen    No rules          52     59     43        58         1         213  
------  ----------------  -

In [29]:
print("the same counts split by state type\n")
for m in MODELS:
    for c in ("full", "nowidth", "norules"):
        onzero, onpick = collections.Counter(), collections.Counter()
        for r in rows(m, c):
            vc = r.get("violation_cause")
            if not vc:
                continue
            (onzero if r.get("zero_legal") else onpick)[vc] += 1
        print("%-7s %-9s picking %-48s no-legal-arm %s"
              % (LABEL[m], c, dict(onpick) or "none", dict(onzero) or "none"))
    print()

the same counts split by state type

Gemini  full      picking none                                             no-legal-arm {'no_route': 7}
Gemini  nowidth   picking {'grasp': 80}                                    no-legal-arm {'grasp': 53}
Gemini  norules   picking none                                             no-legal-arm {'grasp': 1}

GPT     full      picking {'no_route': 2, 'grasp': 5}                      no-legal-arm {'no_route': 8, 'grasp': 2, 'delicate': 2}
GPT     nowidth   picking {'grasp': 74}                                    no-legal-arm {'no_route': 3, 'grasp': 47}
GPT     norules   picking {'grasp': 13}                                    no-legal-arm {'no_route': 5, 'grasp': 15, 'delicate': 3}

Qwen    full      picking {'arm_state': 26, 'reach': 21, 'grasp': 14, 'delicate': 30, 'no_route': 1} no-legal-arm {'grasp': 27, 'arm_state': 41, 'reach': 31, 'delicate': 9}
Qwen    nowidth   picking {'arm_state': 20, 'reach': 18, 'grasp': 16, 'delicate': 26, 'no_route': 1} 

---

# T7. Calibration: does the model know what it lost?

The synthesis. Three signatures, all Full information against Width removed.

1. **Does it say so?** Grasp-error reasons that admit the width is missing.
2. **Does it act so?** Correct refusal. A model that noticed it had less
   information should decline *more*.
3. **What does it do instead?** Franka share. Two Franka and two UR, so 50 is
   indifference between the arm types.

Two things the cells enforce. The reasons bound is reported **per model**,
because the denominators differ by a factor of three. And the refusal signature
is **not applicable to Qwen**, which refuses on nothing at either level, so the
calibration argument is a two-model claim with Qwen explained separately.

In [30]:
import re
# Deliberately generous: a loose pattern that still finds nothing is stronger
# evidence than a strict one that finds nothing.
ADMITS = re.compile(
    r"(unknown|unstated|not stated|missing|unspecified|no (?:declared )?width"
    r"|not (?:given|provided|specified|declared)|absent)", re.I)

body = []
for m in MODELS:
    # 1. says so, counted over both width-absent cells of Design 1
    adm = ex = 0
    for c in ("nowidth", "nowidth-anon"):
        for r in rows(m, c):
            if r.get("violation_cause") != "grasp":
                continue
            ex += 1
            adm += bool(ADMITS.search(r.get("model_reason") or ""))
    bound = "%.1f%%" % (100 * 3 / ex) if (adm == 0 and ex) else "n/a"

    # 2. acts so, at the scene denominator, which is authoritative
    rb, nb = refusal_scene(rows(m, "full"))
    rm, nm = refusal_scene(rows(m, "nowidth"))

    # 3. what instead
    fb, nfb = franka(rows(m, "full"))
    fm, nfm = franka(rows(m, "nowidth"))
    lo, hi = wilson(fm, nfm)

    body.append([LABEL[m], "Reasons admitting the gap",
                 "--", "%d/%d" % (adm, ex), "upper bound " + bound])
    body.append(["", "Correct refusal (scene)", "%.1f" % pct(rb, nb),
                 "%.1f" % pct(rm, nm), gap((rb, nb), (rm, nm))])
    body.append(["", "Franka share", "%.1f" % pct(fb, nfb),
                 ci(fm, nfm), gap((fm, nfm), (fb, nfb))])

table(["Model", "Signature", "Width shown", "Width removed", "Change [95% CI]"],
      body, "T7  Three signatures of an unregistered loss")
print("\nRefusal change is Full information minus Width removed, so a POSITIVE")
print("value means the model abstains LESS when it knows less.")


T7  Three signatures of an unregistered loss
------  -------------------------  -----------  -----------------  --------------------
Model   Signature                  Width shown  Width removed      Change [95% CI]     
------  -------------------------  -----------  -----------------  --------------------
Gemini  Reasons admitting the gap  --           0/256              upper bound 1.2%    
        Correct refusal (scene)    91.7         50.0               +41.7 [+21.1, +58.1]
        Franka share               28.1         48.0 [43.3, 52.7]  +20.0 [+13.4, +26.3]
GPT     Reasons admitting the gap  --           0/237              upper bound 1.3%    
        Correct refusal (scene)    88.9         52.8               +36.1 [+15.3, +53.2]
        Franka share               26.0         45.7 [41.0, 50.6]  +19.7 [+13.1, +26.1]
Qwen    Reasons admitting the gap  --           0/85               upper bound 3.5%    
        Correct refusal (scene)    0.0          0.0                +0.0 [-

In [31]:
# Signature 1, per model. The pooled bound flatters Qwen's much smaller
# denominator and must not stand in for all three.
print("Bound on reasons admitting the missing width\n")
ta = te = 0
for m in MODELS:
    adm = ex = 0
    for c in ("nowidth", "nowidth-anon"):
        for r in rows(m, c):
            if r.get("violation_cause") != "grasp":
                continue
            ex += 1
            adm += bool(ADMITS.search(r.get("model_reason") or ""))
    ta += adm; te += ex
    print("  %-7s %d of %3d, one-sided 95%% upper bound %.1f%%"
          % (LABEL[m], adm, ex, 100 * 3 / ex))
print("  %-7s %d of %3d, one-sided 95%% upper bound %.2f%%" % ("pooled", ta, te, 100 * 3 / te))

Bound on reasons admitting the missing width

  Gemini  0 of 256, one-sided 95% upper bound 1.2%
  GPT     0 of 237, one-sided 95% upper bound 1.3%
  Qwen    0 of  85, one-sided 95% upper bound 3.5%
  pooled  0 of 578, one-sided 95% upper bound 0.52%


In [32]:
# Signature 2 at both denominators. tab:ex1:signatures reports scene and the
# older notes quote trial. Both are correct and they differ, so print them
# together and never quote one under the other's label.
body = []
for m in MODELS:
    at, bt = refusal(rows(m, "full")), refusal(rows(m, "nowidth"))
    a,  b  = refusal_scene(rows(m, "full")), refusal_scene(rows(m, "nowidth"))
    note = "refuses on nothing at either level, signature not applicable" \
           if (at[0] == 0 and bt[0] == 0) else ""
    body.append([LABEL[m], "%.1f" % pct(*at), "%.1f" % pct(*bt), gap(at, bt),
                 "%.1f" % pct(*a), "%.1f" % pct(*b), gap(a, b), note])
table(["Model", "trial: full", "trial: nowidth", "trial change",
       "scene: full", "scene: nowidth", "scene change", "Note"], body,
      "T7b  Correct refusal at both denominators")


T7b  Correct refusal at both denominators
------  -----------  --------------  --------------------  -----------  --------------  --------------------  ------------------------------------------------------------
Model   trial: full  trial: nowidth  trial change          scene: full  scene: nowidth  scene change          Note                                                        
------  -----------  --------------  --------------------  -----------  --------------  --------------------  ------------------------------------------------------------
Gemini  93.5         50.9            +42.6 [+31.4, +52.5]  91.7         50.0            +41.7 [+21.1, +58.1]                                                              
GPT     88.9         53.7            +35.2 [+23.5, +45.6]  88.9         52.8            +36.1 [+15.3, +53.2]                                                              
Qwen    0.0          0.0             +0.0 [-3.4, +3.4]     0.0          0.0             +0.0 [-9.6, +9

In [33]:
# Signature 3. "As if every object fits" is a measurable claim, not a figure of
# speech: it predicts indifference between the arm types, which is 50 per cent.
print("Franka share with the width removed, against 50.0\n")
for m in MODELS:
    k, n = franka(rows(m, "nowidth"))
    lo, hi = wilson(k, n)
    print("  %-7s %5.1f [%5.1f, %5.1f]  %s"
          % (LABEL[m], pct(k, n), lo, hi,
             "INCLUDES 50.0, indistinguishable from indifference"
             if lo <= 50.0 <= hi else "excludes 50.0"))

Franka share with the width removed, against 50.0

  Gemini   48.0 [ 43.3,  52.7]  INCLUDES 50.0, indistinguishable from indifference
  GPT      45.7 [ 41.0,  50.6]  INCLUDES 50.0, indistinguishable from indifference
  Qwen     21.8 [ 18.4,  25.7]  excludes 50.0


In [34]:
# A sample of the reason strings, so "not one admits the gap" is inspectable
# rather than taken on trust from a regular expression.
seen = 0
for r in rows("gpt", "nowidth"):
    if r.get("violation_cause") != "grasp":
        continue
    print("-", (r.get("model_reason") or "")[:130])
    seen += 1
    if seen >= 8:
        break

- Franka_s can directly deliver the clamp within sw while preserving ur_w for the wood block.
- Franka_s directly handles the clamp and preserves ur_w for the wood block.
- Franka_s can directly deliver the clamp while preserving ur_w for exclusive heavy tasks.
- Franka_s can deliver the clamp directly while preserving ur_w for exclusive tasks.
- franka_s can directly move the large clamp to the tools basket.
- Franka_s can deliver the clamp directly, preserving ur_w for uniquely reachable tasks.
- Franka_s can deliver the clamp directly while preserving ur_w for exclusive tasks.
- Franka_s can directly deliver the clamp to the tools destination.


---

# T8. Cast B

Not in the plan. Included because object-set generalisation was a supervisor
requirement and the result partly fails, which is a finding rather than an
inconvenience. Direction reproduces, magnitude does not.

The single-repeat files are an independent execution of the same cells at the
same prompt version, not a subset of the three-repeat run, so they are a
run-to-run stability check and must not be pooled into the headline figures.

In [35]:
def rows_b(cond, r3=True):
    fn = "ex1_castb_gpt_%s_r%d.jsonl" % (cond, 3 if r3 else 1)
    with open(os.path.join(ROOT, "out", fn)) as f:
        return [json.loads(l) for l in f if l.strip()]

floors_b = json.load(open(os.path.join(ROOT, "out", "ex1_setb_floors.json")))
WB_B = 100 * floors_b["by_cause"]["grasp"]["width_blind"]["mean"]

body = []
base = legality(rows_b("full"))
for c in ("full", "nowidth", "nowidth-anon"):
    k, n = legality(rows_b(c))
    ki, ni = legality(rows_b(c, r3=False))
    body.append([PRETTY[c], ci(k, n), "%.1f" % pct(ki, ni),
                 "--" if c == "full" else gap(base, (k, n))])
table(["Condition", "Legality (3 repeats)", "Independent single run",
       "Gap from full information"], body, "T8  Cast B, GPT only, 108 states")

a = newcombe(*legality(rows("gpt", "full")), *legality(rows("gpt", "nowidth")))
b = newcombe(*legality(rows_b("full")), *legality(rows_b("nowidth")))
print("\nwidth-removal gap, GPT")
print("  cast A  %+.1f [%+.1f, %+.1f]   width-blind line %.1f" % (a + (WIDTH_BLIND,)))
print("  cast B  %+.1f [%+.1f, %+.1f]   width-blind line %.1f" % (b + (WB_B,)))
print("  both exclude zero: %s     intervals overlap: %s"
      % (not spans_zero(a) and not spans_zero(b), not (a[2] < b[1] or b[2] < a[1])))


T8  Cast B, GPT only, 108 states
----------------  --------------------  ----------------------  -------------------------
Condition         Legality (3 repeats)  Independent single run  Gap from full information
----------------  --------------------  ----------------------  -------------------------
Full information  98.8 [95.8, 99.7]     98.1                    --                       
Width removed     90.1 [84.6, 93.8]     88.5                    +8.7 [+3.9, +14.3]       
Both removed      88.5 [82.7, 92.5]     92.2                    +10.3 [+5.3, +16.2]      
----------------  --------------------  ----------------------  -------------------------

width-removal gap, GPT
  cast A  +24.0 [+18.5, +29.6]   width-blind line 74.9
  cast B  +8.7 [+3.9, +14.3]   width-blind line 69.2
  both exclude zero: True     intervals overlap: False


In [36]:
# Exposure counts, which the "not an exposure artefact" paragraph needs now that
# the per-object table is gone.
setb = json.load(open(os.path.join(ROOT, "probes", "ex1_setb_v1.json")))["probes"]
opps = collections.Counter()
true_b = {}
for p in setb:
    pv = p["provenance"]
    for t in p["state"]["tasks"]:
        opps[t["object"]] += 1
        true_b[(pv["source"], pv["seq"], pv["round"], t["id"])] = t["object"]

chosen, errs = collections.Counter(), collections.Counter()
for r in rows_b("nowidth"):
    tid = (r.get("decision") or {}).get("task_id")
    if tid is None:
        continue
    pv = r["provenance"]
    o = true_b.get((pv["source"], pv["seq"], pv["round"], tid))
    if o is None:
        continue
    chosen[o] += 1
    errs[o] += r.get("result") == "rejected"

table(["Object", "States offering it", "Times chosen", "Errors"],
      [[o.replace("ycb_", ""), opps[o], chosen[o], errs[o]]
       for o in sorted(opps, key=lambda k: -opps[k])],
      "T8b  Cast B exposure, GPT, width removed")


T8b  Cast B exposure, GPT, width removed
-------------  ------------------  ------------  ------
Object         States offering it  Times chosen  Errors
-------------  ------------------  ------------  ------
caster         101                 23            1     
sugar_box      83                  43            16    
t_connector    82                  51            0     
scissors       69                  43            3     
tuna_can       65                  10            0     
bracket_small  61                  47            0     
bleach         60                  5             3     
screw_99       57                  16            0     
foam_brick     52                  17            1     
mac_n_cheese   37                  14            0     
-------------  ------------------  ------------  ------


---

# Cross-check against `ex1_results.py`

Everything above was computed here, independently of the pipeline. That is the
point of the notebook, and it is also the risk: two implementations can drift.
This cell imports the pipeline and compares the load-bearing contrasts. Any row
that does not agree means one of the two is wrong.

In [37]:
for cand in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
    if (cand / "ex1_results.py").exists():
        sys.path.insert(0, str(cand)); break
    if (cand / "analysis" / "ex1" / "ex1_results.py").exists():
        sys.path.insert(0, str(cand / "analysis" / "ex1")); break

try:
    import ex1_results as R
    FILES = R.resolve_files(R.find_root())[0]
    ok = bad = 0
    print("%-7s %-28s %-22s %-22s" % ("model", "contrast", "here", "pipeline"))
    for m in MODELS:
        for label, base, manip in (("width removed, name kept", "full", "nowidth"),
                                   ("width removed, anonymised", "anon", "nowidth-anon"),
                                   ("name removed, width kept", "full", "anon"),
                                   ("name swapped, width kept", "full", "swap"),
                                   ("rules removed", "full", "norules")):
            mine = newcombe(*legality(rows(m, base)), *legality(rows(m, manip)))
            theirs = R.newcombe(*R.legality(R.load(R.find_root(), FILES[("casta", m, base)])),
                                *R.legality(R.load(R.find_root(), FILES[("casta", m, manip)])))
            agree = all(abs(x - y) < 0.05 for x, y in zip(mine, theirs))
            ok += agree; bad += not agree
            print("%-7s %-28s %+6.1f [%+5.1f,%+5.1f]  %+6.1f [%+5.1f,%+5.1f]  %s"
                  % (LABEL[m], label, mine[0], mine[1], mine[2],
                     theirs[0], theirs[1], theirs[2], "" if agree else "MISMATCH"))
    print("\n%d agree, %d disagree" % (ok, bad))
except ImportError as exc:
    print("ex1_results.py not importable (%s). The tables above still stand on"
          " their own, but the cross-check has not run." % exc)

model   contrast                     here                   pipeline              
Gemini  width removed, name kept      +27.8 [+22.7,+33.2]   +27.8 [+22.7,+33.2]  
Gemini  width removed, anonymised     +24.7 [+19.8,+29.9]   +24.7 [+19.8,+29.9]  
Gemini  name removed, width kept       +0.0 [ -1.3, +1.3]    +0.0 [ -1.3, +1.3]  
Gemini  name swapped, width kept       +0.0 [ -1.3, +1.3]    +0.0 [ -1.3, +1.3]  
Gemini  rules removed                  +0.0 [ -1.3, +1.3]    +0.0 [ -1.3, +1.3]  
GPT     width removed, name kept      +24.0 [+18.5,+29.6]   +24.0 [+18.5,+29.6]  
GPT     width removed, anonymised     +21.9 [+16.8,+27.2]   +21.9 [+16.8,+27.2]  
GPT     name removed, width kept       -1.0 [ -3.7, +1.5]    -1.0 [ -3.7, +1.5]  
GPT     name swapped, width kept       +0.1 [ -2.8, +3.0]    +0.1 [ -2.8, +3.0]  
GPT     rules removed                  +2.4 [ -0.9, +5.9]    +2.4 [ -0.9, +5.9]  
Qwen    width removed, name kept       -1.4 [ -8.3, +5.6]    -1.4 [ -8.3, +5.6]  
Qwen    width r

---

## What this notebook does not decide

1. Whether the baseline and Legal-Arm Control get their own table. The plan has
   no baseline, and the width gap has nothing to be a gap from without one.
2. Whether cast B (T8) stays in the results or moves to an appendix.
3. The one errored row in `ex1_casta_gemini_nowidth-swap_r3.jsonl`, 1 of 486,
   which makes that cell n=287. Footnote it or repair it.
4. Legal-Arm Control is one repeat while everything else is three.
5. Whether the swap refusal contrasts in T3b go in the chapter, or the plan's
   promise to report refusal for Design 2 is dropped.